In [1]:

from pathlib import Path
import random, csv, json, math, statistics, re

CONFIG = {
    "seed": 42,
    "output_root": "ga_partition_output_corrected",
    "population_size": 20,
    "generations": 20,
    "mutation_rate": 0.15,
    "crossover_rate": 0.80,
    "elitism": 2,
    "top_models_to_export": 10,
    "valid_bonus": 10000,
    "solution_bonus": 500,
    "branch_penalty": 1.0,
    "step_penalty": 0.1,
    "cases": [
        {"case_id": 1, "weights": [1,2,3], "target": 3},
        {"case_id": 2, "weights": [1,2,3,4], "target": 5},
        {"case_id": 3, "weights": [1,2,3,4,5], "target": 7},
        {"case_id": 4, "weights": [1,2,3,4,5,6], "target": 10},
        {"case_id": 5, "weights": [1,2,3,4,5,6,7,8], "target": 18},
        {"case_id": 6, "weights": [1,2,3,4,5,6,7,8,9,10], "target": 27},
    ],
}
random.seed(CONFIG["seed"])

ROOT = Path(CONFIG["output_root"])
CAND_DIR = ROOT / "ga_partition_candidates"
RES_DIR = ROOT / "ga_results"
LOG_DIR = ROOT / "ga_logs_manual_kpworkbench"
for d in [ROOT, CAND_DIR, RES_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)
(ROOT / "ga_config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

def kp_comment(text):
    return "/* " + str(text).replace("*/", "* /") + " */"

def pruned_metrics(order, target):
    mult = {0: 1}
    for w in order:
        nxt = {}
        for s, cnt in mult.items():
            nxt[s] = nxt.get(s, 0) + cnt
            if s + w <= target:
                nxt[s+w] = nxt.get(s+w, 0) + cnt
        mult = nxt
    branches = sum(mult.values())
    sol = mult.get(target, 0)
    exhaustive = 2 ** len(order)
    return {
        "valid": 1,
        "has_solution": int(sol > 0),
        "terminal_branches": branches,
        "solution_branches": sol,
        "failure_branches": branches - sol,
        "exhaustive_branches": exhaustive,
        "branch_reduction": 1 - branches / exhaustive,
        "estimated_steps": len(order) + 2,
    }

def fitness(order, target):
    m = pruned_metrics(order, target)
    score = (CONFIG["valid_bonus"] * m["valid"]
             + CONFIG["solution_bonus"] * m["has_solution"]
             - CONFIG["branch_penalty"] * m["terminal_branches"]
             - CONFIG["step_penalty"] * m["estimated_steps"])
    return score, m

def init_pop(weights):
    pop, seen = [], set()
    for c in [weights[:], list(reversed(weights)), sorted(weights), sorted(weights, reverse=True)]:
        t = tuple(c)
        if t not in seen:
            pop.append(c); seen.add(t)
    max_perm = math.factorial(len(weights))
    while len(pop) < CONFIG["population_size"] and len(seen) < max_perm:
        c = weights[:]
        random.shuffle(c)
        t = tuple(c)
        if t not in seen:
            pop.append(c); seen.add(t)
    while len(pop) < CONFIG["population_size"]:
        pop.append(random.choice(pop)[:])
    return pop

def order_crossover(p1, p2):
    n = len(p1)
    if n < 3:
        return p1[:]
    a, b = sorted(random.sample(range(n), 2))
    child = [None] * n
    child[a:b+1] = p1[a:b+1]
    rest = [x for x in p2 if x not in child]
    j = 0
    for i in range(n):
        if child[i] is None:
            child[i] = rest[j]; j += 1
    return child

def mutate(c):
    c = c[:]
    if random.random() < CONFIG["mutation_rate"] and len(c) > 1:
        i, j = random.sample(range(len(c)), 2)
        c[i], c[j] = c[j], c[i]
    return c

def select(scored):
    g = random.sample(scored, min(3, len(scored)))
    g.sort(key=lambda x: x[0], reverse=True)
    return g[0][1][:]

def generate_kplt(case_id, order, target, cand_id):
    n = len(order)
    include, exclude = [], []
    for i, w in enumerate(order, 1):
        next_stage = f"a{i+1}" if i < n else "x"
        for s in range(target + 1):
            if s + w <= target:
                include.append(
                    f"    =a{i} & =p{s} : a{i}, p{s} -> p{s+w}, {next_stage} . "
                    f"{kp_comment('include item ' + str(i) + ', weight ' + str(w))}"
                )
            exclude.append(
                f"    =a{i} & =p{s} : a{i}, p{s} -> p{s}, {next_stage} . "
                f"{kp_comment('exclude item ' + str(i) + ', weight ' + str(w))}"
            )
    terminal = [f"    =x & =p{target} : x, p{target} -> t (T1) . {kp_comment('success')}"]
    for s in range(target + 1):
        if s != target:
            terminal.append(f"    =x & =p{s} : x, p{s} -> f (T1) . {kp_comment('failure')}")
    head = kp_comment(
        f"GA candidate {cand_id}; case={case_id}; n={n}; target={target}; "
        f"order={order}; stage markers and terminal markers are consumed"
    )
    return f"""{head}

#define n = {n}
#define k = {target}

type T0 {{
}}

type T1 {{
max {{
    >=t : s -> yes (T0) .
    >=f & <t : s -> no (T0) .
}}
}}

type T2 {{
{kp_comment('Include branches')}
choice {{
{chr(10).join(include)}
}}

{kp_comment('Exclude branches')}
choice {{
{chr(10).join(exclude)}
}}

{kp_comment('Terminal decision')}
choice {{
{chr(10).join(terminal)}
}}
}}

t0 {{}} (T0) .
t1 {{s}} (T1) .
t2 {{a1, p0}} (T2) .
t0 - t1 .
t1 - t2 .
"""

def write_csv(path, rows):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)

all_best, all_hist, all_chrom = [], [], []
for case in CONFIG["cases"]:
    cid, weights, target = case["case_id"], case["weights"], case["target"]
    pop = init_pop(weights)
    best_global = None
    for gen in range(CONFIG["generations"]):
        scored = []
        for chrom in pop:
            fit, met = fitness(chrom, target)
            scored.append((fit, chrom, met))
        scored.sort(key=lambda x: x[0], reverse=True)
        best_fit, best_chrom, best_met = scored[0]
        avg_fit = statistics.mean(x[0] for x in scored)
        all_hist.append({
            "case_id": cid, "generation": gen,
            "best_fitness": round(best_fit, 4),
            "avg_fitness": round(avg_fit, 4),
            "best_chromosome": " ".join(map(str, best_chrom)),
            "best_terminal_branches": best_met["terminal_branches"],
            "best_solution_branches": best_met["solution_branches"],
            "best_branch_reduction": round(best_met["branch_reduction"], 6),
        })
        if best_global is None or best_fit > best_global[0]:
            best_global = (best_fit, best_chrom[:], dict(best_met), gen)
        new_pop = [x[1][:] for x in scored[:CONFIG["elitism"]]]
        while len(new_pop) < CONFIG["population_size"]:
            p1, p2 = select(scored), select(scored)
            child = order_crossover(p1, p2) if random.random() < CONFIG["crossover_rate"] else p1[:]
            new_pop.append(mutate(child))
        pop = new_pop[:CONFIG["population_size"]]
    final = []
    for chrom in pop + [best_global[1]]:
        fit, met = fitness(chrom, target)
        final.append((fit, chrom, met))
    final.sort(key=lambda x: x[0], reverse=True)
    seen, rank = set(), 1
    for fit, chrom, met in final:
        if tuple(chrom) in seen:
            continue
        seen.add(tuple(chrom))
        cand_id = f"case{cid:02d}_cand{rank:03d}"
        model_path = CAND_DIR / f"{cand_id}.kplt"
        model_path.write_text(generate_kplt(cid, chrom, target, cand_id), encoding="utf-8")
        all_chrom.append({
            "case_id": cid, "candidate_id": cand_id, "model_file": str(model_path),
            "fitness": round(fit, 4), "chromosome": " ".join(map(str, chrom)),
            "target": target, "terminal_branches_proxy": met["terminal_branches"],
            "solution_branches_proxy": met["solution_branches"],
            "failure_branches_proxy": met["failure_branches"],
            "exhaustive_branches": met["exhaustive_branches"],
            "branch_reduction_proxy": round(met["branch_reduction"], 6),
            "estimated_steps": met["estimated_steps"],
        })
        rank += 1
        if rank > CONFIG["top_models_to_export"]:
            break
    fit, chrom, met, gen = best_global
    all_best.append({
        "case_id": cid, "n": len(weights), "target": target,
        "original_weights": " ".join(map(str, weights)),
        "best_generation": gen, "best_fitness": round(fit, 4),
        "best_chromosome": " ".join(map(str, chrom)),
        "best_terminal_branches_proxy": met["terminal_branches"],
        "best_solution_branches_proxy": met["solution_branches"],
        "best_failure_branches_proxy": met["failure_branches"],
        "exhaustive_branches": met["exhaustive_branches"],
        "branch_reduction_proxy": round(met["branch_reduction"], 6),
        "estimated_steps": met["estimated_steps"],
    })

write_csv(RES_DIR / "best_candidates.csv", all_best)
write_csv(RES_DIR / "fitness.csv", all_hist)
write_csv(RES_DIR / "chromosomes.csv", all_chrom)

print("Completed corrected GA workflow.")
print("Generated .kplt comments use /* ... */ syntax.")
print("Output:", ROOT.resolve())
print("Candidate .kplt files:", len(list(CAND_DIR.glob("*.kplt"))))
print("CSV:", RES_DIR / "best_candidates.csv", RES_DIR / "fitness.csv", RES_DIR / "chromosomes.csv")
for r in all_best:
    print(f"Case {r['case_id']}: order=[{r['best_chromosome']}], "
          f"branches={r['best_terminal_branches_proxy']}/{r['exhaustive_branches']}, "
          f"reduction={100*r['branch_reduction_proxy']:.2f}%, "
          f"solutions={r['best_solution_branches_proxy']}")


Completed corrected GA workflow.
Generated .kplt comments use /* ... */ syntax.
Output: C:\Users\student\ICMC2026\ga_partition_output_corrected
Candidate .kplt files: 54
CSV: ga_partition_output_corrected\ga_results\best_candidates.csv ga_partition_output_corrected\ga_results\fitness.csv ga_partition_output_corrected\ga_results\chromosomes.csv
Case 1: order=[1 2 3], branches=5/8, reduction=37.50%, solutions=2
Case 2: order=[1 2 3 4], branches=9/16, reduction=43.75%, solutions=2
Case 3: order=[1 2 3 4 5], branches=16/32, reduction=50.00%, solutions=3
Case 4: order=[1 2 3 4 5 6], branches=32/64, reduction=50.00%, solutions=5
Case 5: order=[1 2 3 4 5 6 7 8], branches=135/256, reduction=47.27%, solutions=14
Case 6: order=[1 2 3 4 5 6 7 8 9 10], branches=512/1024, reduction=50.00%, solutions=40
